# Entrenamiento de una LSTM para la detección de anomalías en producción solar

- **Dataset:** https://www.kaggle.com/datasets/anikannal/solar-power-generation-data/data
- **Modelo:** LSTM Autoencoder

## 1. Dependencias y configuración inicial

In [1]:
!nvidia-smi

In [2]:
# Python
import hashlib
import json
import os
import random
import shutil
import time

import joblib
import kagglehub

# Visualización
import matplotlib.pyplot as plt

# Manipulación de datos y utilidades generales
import numpy as np
import pandas as pd
import seaborn as sns

# DL (TensorFlow y Keras)
import tensorflow as tf
from keras.callbacks import EarlyStopping, ReduceLROnPlateau
from keras.layers import (
    LSTM,
    Bidirectional,
    Dense,
    Dropout,
    Input,
    LayerNormalization,
    RepeatVector,
    TimeDistributed,
)
from keras.models import Model, Sequential
from keras.optimizers import Adam
from keras.utils import plot_model
from sklearn.ensemble import IsolationForest
from sklearn.linear_model import LinearRegression
from sklearn.metrics import auc, precision_recall_curve, roc_auc_score

# ML (Scikit-Learn)
# from sklearn.preprocessing import RobustScaler
from sklearn.preprocessing import MinMaxScaler, minmax_scale

# Google Colab
# Deteccion entorno
try:
    from google.colab import drive  # type: ignore[import-untyped]

    IN_COLAB = True
except ImportError:
    IN_COLAB = False

In [3]:
# Reproducibilidad
def set_seeds(seed=42):
    # Python
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    # NumPy
    np.random.seed(seed)
    # TensorFlow
    tf.random.set_seed(seed)
    # Para operaciones en GPU
    os.environ["TF_DETERMINISTIC_OPS"] = "1"
    os.environ["TF_CUDNN_DETERMINISTIC"] = "1"


set_seeds(42)

In [4]:
# Configuración básica para visualizaciones
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (12, 6)

## 2. Importación del dataset

In [5]:
path = kagglehub.dataset_download("anikannal/solar-power-generation-data")
print("Datasets disponibles:", os.listdir(path))

available_datasets = os.listdir(path)
datasets = {}

for file in available_datasets:
    file_path = os.path.join(path, file)
    datasets[file] = pd.read_csv(file_path)

In [6]:
# Utilidad para generar un contexto LLM-friendly
# Esto ayuda a refinar nuestro análisis con ayuda de IA Generativa
def generate_datasets_context(dataframes: dict) -> str:
    context = (
        "A continuación se detalla la estructura y una muestra de los datasets disponibles:\n\n"
    )

    for name, df in dataframes.items():
        context += f"### Dataset: `{name}`\n"
        context += f"- **Shape**: {df.shape[0]} filas, {df.shape[1]} columnas\n"

        # Agrupamos columnas y tipos en un formato tipo JSON
        types = df.dtypes.astype(str).to_dict()
        types_str = ", ".join([f"'{col}': {tipo}" for col, tipo in types.items()])
        context += f"- **Columnas y Tipos**: {{{types_str}}}\n"

        context += "- **Muestra (primeras 3 filas)**:\n"
        context += "```csv\n"
        context += df.head(3).to_csv(index=False)
        context += "```\n\n"

    return context

In [7]:
context = generate_datasets_context(datasets)
print(context)

In [8]:
# Cargas en memoria de nuestos datasets
datasets = {f: pd.read_csv(os.path.join(path, f)) for f in os.listdir(path)}

df_gen_1 = datasets["Plant_1_Generation_Data.csv"]
df_wea_1 = datasets["Plant_1_Weather_Sensor_Data.csv"]
df_gen_2 = datasets["Plant_2_Generation_Data.csv"]
df_wea_2 = datasets["Plant_2_Weather_Sensor_Data.csv"]

## 3. Correcciones preliminares

In [9]:
# Estandarizado en el formato de fechas
# Nota: Los datasets NO tienen un formato de fecha homogéneo.
# - Plant_1_Generation_Data usa formato europeo: '%d-%m-%Y %H:%M'
# - El resto de datasets usan formato estándar ISO: '%Y-%m-%d %H:%M:%S'

# Planta 1
# Corregir fechas
df_gen_1["DATE_TIME"] = pd.to_datetime(df_gen_1["DATE_TIME"], format="%d-%m-%Y %H:%M")
df_wea_1["DATE_TIME"] = pd.to_datetime(df_wea_1["DATE_TIME"], format="%Y-%m-%d %H:%M:%S")

# Planta 2
# Corregir fechas
df_gen_2["DATE_TIME"] = pd.to_datetime(df_gen_2["DATE_TIME"], format="%Y-%m-%d %H:%M:%S")
df_wea_2["DATE_TIME"] = pd.to_datetime(df_wea_2["DATE_TIME"], format="%Y-%m-%d %H:%M:%S")

## 4. Unificación de los datos

In [10]:
# Fusionar datasets de planta 1 y 2 (generation + weather)
df_p1 = pd.merge(
    df_gen_1,
    df_wea_1.drop(columns=["PLANT_ID", "SOURCE_KEY"]),
    on="DATE_TIME",
    how="inner",
)
df_p2 = pd.merge(
    df_gen_2,
    df_wea_2.drop(columns=["PLANT_ID", "SOURCE_KEY"]),
    on="DATE_TIME",
    how="inner",
)

print(f"Dataset Planta 1 unificado: {df_p1.shape}")
print(f"Dataset Planta 2 unificado: {df_p2.shape}")

In [11]:
# Verificación de que el inner join no descartó registros de generación
# Si un timestamp de generación no tiene medición meteorológica se pierde
n_dropped_p1 = len(df_gen_1) - len(df_p1)
n_dropped_p2 = len(df_gen_2) - len(df_p2)

if n_dropped_p1 > 0 or n_dropped_p2 > 0:
    print(f"El merge descartó {n_dropped_p1} filas de Planta 1 y {n_dropped_p2} de Planta 2")
else:
    print("El merge no descartó ningún registro de generación")

# También verificamos que el sensor meteorológico no tenga timestamps duplicados
# Un timestamp duplicado en weather inflaría el join
dup_wea_1 = df_wea_1["DATE_TIME"].duplicated().sum()
dup_wea_2 = df_wea_2["DATE_TIME"].duplicated().sum()
print(f"Timestamps duplicados en weather Planta 1: {dup_wea_1}")
print(f"Timestamps duplicados en weather Planta 2: {dup_wea_2}")

In [12]:
df_p1["PLANT"] = 1
df_p2["PLANT"] = 2

df_full = pd.concat([df_p1, df_p2], ignore_index=True)
# Ordenamos los registros por planta, inversor y fecha
df_full = df_full.sort_values(by=["PLANT", "SOURCE_KEY", "DATE_TIME"]).reset_index(drop=True)

print(f"Dimensiones del Dataset Final (df_full): {df_full.shape}")

context_full = generate_datasets_context({"df_full": df_full})
print(context_full)

## 5. EDA

In [13]:
print("¿Valores nulos en Planta 1?")
print(df_p1.isnull().sum())

print("\n¿Valores nulos en Planta 2?")
print(df_p2.isnull().sum())

In [14]:
# Comprobación de registros duplicados por (DATE_TIME, SOURCE_KEY)
# Un duplicado aquí significaría dos lecturas del mismo inversor en el mismo instante
dup_p1 = df_p1.duplicated(subset=["DATE_TIME", "SOURCE_KEY"]).sum()
dup_p2 = df_p2.duplicated(subset=["DATE_TIME", "SOURCE_KEY"]).sum()

print(f"Duplicados (DATE_TIME, SOURCE_KEY) en Planta 1: {dup_p1}")
print(f"Duplicados (DATE_TIME, SOURCE_KEY) en Planta 2: {dup_p2}")

In [15]:
# Estadí­sticas generales separadas por planta
print("Estadísticas descriptivas para Planta 1:")
display(df_full[df_full["PLANT"] == 1].describe())

print("\nEstadísticas descriptivas para Planta 2:")
display(df_full[df_full["PLANT"] == 2].describe())

### 5.1 Observación de los estadísticos

##### A) Anomalías en Potencia DC vs Potencia AC

- **Planta 1:** Tiene un `DC_POWER` alto (máx. 14.471,13) comparado con su `AC_POWER` (máx. 1.410,95). Hay un factor de aproximadamente 10 a 1.
- **Planta 2:** Sus valores de `DC_POWER` (máx. 1.420,93) y `AC_POWER` (máx. 1.385,42) están casi a la par (relación 1 a 1).

**¿La Planta 1 genera 10 veces más energía que la planta 2?**

No. El `DAILY_YIELD` (rendimiento diario) y el `AC_POWER` de ambas plantas son bastante similares. Esto indica que los sensores de `DC_POWER` de la Planta 1 y la Planta 2 están midiendo en diferentes unidades o escalas (por ejemplo, vatios frente a kilovatios, o registros de inversores en distintas configuraciones).

> **Nota:** No podremos poner el `DC_POWER` de ambas plantas en el mismo eje de un gráfico sin normalizarlos primero, o el gráfico quedará totalmente distorsionado. Para el modelo, corregiremos la escala de la Planta 1 dividiéndola por 10.

##### B) Clima y Condiciones

- **Irradiación:** Ambas plantas reciben niveles de luz solar (`IRRADIATION`) muy similares (media de 0,23 en la Planta 1 y 0,23 en la Planta 2), lo cual tiene sentido si están en la misma región.
- **Temperatura:** La Planta 2 es ligeramente más calurosa. Su temperatura ambiente máxima roza los 39,18°C, frente a los 35,25°C de la Planta 1. La temperatura de los módulos en ambas plantas supera los 65°C en sus picos máximos, lo cual es normal pero un punto clave a vigilar, ya que el calor excesivo reduce la eficiencia del panel.

##### C) Integridad de los Datos y Cuartiles (25%, 50%, 75%)

- **Registros Totales:** La Planta 1 tiene 68.774 registros y la Planta 2 tiene 67.698. Comparar ambos totales directamente no tiene sentido. Son plantas distintas y no tienen por qué tener la misma cantidad de registros. El análisis correcto es contrastar cada planta contra su cobertura esperada. Con 34 días, 96 intervalos de 15 min y 22 inversores, lo esperado son unos 71.808 registros por planta. La Planta 2 presenta unos 4.110 registros ausentes concentrados en 4 inversores que sufrieron un corte prolongado de unos 9 días (detectado en el análisis de huecos temporales).
- **Mediana (50%) de `DC_POWER`:** La Planta 1 tiene mediana 428,57 y la Planta 2 tiene mediana 0. Esto no indica necesariamente más desconexiones diurnas en la Planta 2. Ambas plantas tienen aproximadamente el 50% de sus registros en horario nocturno (`IRRADIATION`=0, `DC_POWER`=0). La diferencia entre medianas se explica porque los 4 inversores anómalos de la Planta 2 tienen ausentes (no presentes como cero) sus registros de producción diurna durante el período del corte, lo que reduce proporcionalmente los registros con valores positivos de `DC_POWER` en la Planta 2.
- **Reseteo Diario (25% de `DAILY_YIELD`):** El percentil 25% de `DAILY_YIELD` en la Planta 1 es 0, mientras que en la Planta 2 es 272,75. El rendimiento diario debería resetearse a cero a medianoche y mantenerse en cero hasta que sale el sol. Que el 25% de los datos de la Planta 1 sea cero encaja con las horas nocturnas. En la Planta 2, el primer registro del día (00:00) de al menos un inversor ya muestra `DAILY_YIELD = 9.425`. El contador no se ha reseteado y arrastra el valor acumulado del día anterior. Esto contaminaría la señal si se usase como feature del modelo.

##### D) Rendimientos y varianza histórica (`TOTAL_YIELD`)

- **Valores mínimos:** La Planta 1 empieza en 6.183.645, lo que indica que es una planta más antigua o que sus inversores llevan mucho tiempo sin resetearse. La Planta 2 tiene un mínimo de 0, lo que sugiere que algunos inversores son completamente nuevos o se reiniciaron durante el período de lectura.
- **Valores máximos:** El `TOTAL_YIELD` máximo de la Planta 1 es de 7,8 millones. El de la Planta 2 es de 2.248 millones.
- **Conclusión:** Esta magnitud extrema en la Planta 2, combinada con su mínimo de 0, genera una gran desviación estándar (729 millones). Esto indica una mezcla de inversores con historiales muy distintos o una diferencia en las unidades de medida dentro de la misma planta (Wh vs. kWh).

##### Consecuencia para el modelo

Las variables `DAILY_YIELD` y `TOTAL_YIELD` se **excluyen de las features de entrenamiento** como consecuencia directa de estas observaciones. `DAILY_YIELD` tiene un bug de reseteo en la Planta 2 que introduce ruido y `TOTAL_YIELD` presenta inconsistencias de escala y unidades que harían el escalado erróneo. Ambas son variables acumulativas que no aportan valor a la predicción.

### 5.2 Ajustes post-observaciones

In [16]:
# Ajuste posterior a las observaciones

# Normalizar DC_POWER en la Planta 1 (A)
df_full["DC_POWER_NORMALIZED"] = df_full.apply(
    lambda row: row["DC_POWER"] / 10 if row["PLANT"] == 1 else row["DC_POWER"], axis=1
)

# Extraer caracterí­sticas temporales clave para el análisis estacional/diario
df_full["HOUR"] = df_full["DATE_TIME"].dt.hour
df_full["DATE"] = df_full["DATE_TIME"].dt.date

### 5.3 Distribución de variables

In [17]:
# Distribución de las variables de entrada del modelo
# Posiblemente AC_POWER e IRRADIATION tienen una distribución binomial:
#  - Esto es debido a los 2 estados del día: noche y día
plot_features = [
    "DC_POWER_NORMALIZED",
    "AC_POWER",
    "AMBIENT_TEMPERATURE",
    "MODULE_TEMPERATURE",
    "IRRADIATION",
]

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.flatten()

for idx, col in enumerate(plot_features):
    for plant_id, color in zip([1, 2], ["steelblue", "tomato"]):
        subset = df_full[df_full["PLANT"] == plant_id][col]
        sns.histplot(
            subset,
            ax=axes[idx],
            color=color,
            alpha=0.4,
            label=f"Planta {plant_id}",
            kde=True,
            stat="density",
            bins=60,
        )
        axes[idx].axvline(subset.median(), color=color, linestyle="--", alpha=0.6, linewidth=1)
    axes[idx].set_title(col, fontsize=12)
    axes[idx].legend(fontsize=10)
    axes[idx].set_xlabel("")
    axes[idx].set_ylabel("Densidad")

axes[-1].set_visible(False)
fig.suptitle("Distribución de variables por planta (histograma + KDE)", fontsize=15, y=1.01)
plt.tight_layout()
plt.show()

Las distribuciones por planta muestran tres hallazgos: 
1. Tras la corrección de escala, `DC_POWER` y `AC_POWER` siguen distribuciones comparables entre plantas, validando el preprocesado. 
2. Planta 2 opera en un régimen térmico más cálido (ambiental +2.5°C, módulo +3°C), lo que reduce la eficiencia fotovoltaica y explica parcialmente el menor rendimiento. 
3. La bimodalidad de `MODULE_TEMPERATURE` refleja la fase día/noche y justifica la inclusión de `TIME_SIN`/`TIME_COS` como features cíclicas.

In [18]:
plt.figure(figsize=(14, 6))

sns.lineplot(data=df_full, x="HOUR", y="AC_POWER", hue="PLANT", palette="Set1", errorbar="sd")

plt.title("Curva de generación promedio diaria (AC_POWER) por planta", fontsize=16)
plt.xlabel("Hora del día", fontsize=12)
plt.ylabel("Potencia AC", fontsize=12)
plt.xticks(range(0, 24))
plt.legend(title="Planta")
plt.tight_layout()

plt.show()

- Cero eléctrico entre 19-5h: Esto argumenta el uso de un filtro para ventanas nocturnas.
- Producción pico 11-12h:  Este rango horario representa la granja de mayor producción.
- Diferencias entre plantas: 
    - Pico medio **planta 1** aproximado: 950
    - Pico medio **planta 2** aproximado: 650

Un detalle bastante interesante es que la **mayor dispersión de planta 2** durante el mediodía sugiere mayor incidencia de anomalías o heterogeneidad entre inversores.

### 5.4 Matriz de correlación

In [19]:
# Variables clave para correlación
cols_corr = [
    "DC_POWER_NORMALIZED",
    "AC_POWER",
    "AMBIENT_TEMPERATURE",
    "MODULE_TEMPERATURE",
    "IRRADIATION",
]

# Separar por plantas para ver si el clima afecta a una más que a otra
fig, axes = plt.subplots(1, 2, figsize=(18, 7))

for i, plant_id in enumerate([1, 2]):
    corr = df_full[df_full["PLANT"] == plant_id][cols_corr].corr()
    mask = np.tril(np.ones_like(corr, dtype=bool), k=-1)
    sns.heatmap(
        corr,
        mask=mask,
        annot=True,
        cmap="coolwarm",
        vmin=-1,
        vmax=1,
        ax=axes[i],
        square=True,
    )
    axes[i].set_title(f"Matriz de correlación - Planta {plant_id}", fontsize=14)

plt.tight_layout()
plt.show()

- Aunque las correlaciones entre potencias (DC/AC) e irradiación son muy altas, no se eliminan. El autoencoder aprende la relación física entre ellas y las anomalías de interés (por ejemplo un inversor parado con sol) se manifiestan precisamente como rupturas de esas correlaciones (RL != AE). La redundancia, en este caso, no sobra sino que es la detección. Si quitáramos `AC_POWER` por estar correlacionada con `IRRADIATION`, perderíamos la variable que define el fallo.

In [20]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6), sharey=True)

# Planta 1
sns.scatterplot(
    data=df_full[df_full["PLANT"] == 1],
    x="IRRADIATION",
    y="AC_POWER",
    hue="MODULE_TEMPERATURE",
    palette="inferno",
    alpha=0.6,
    edgecolor=None,
    ax=axes[0],
)

axes[0].set_title("Planta 1")
axes[0].set_xlabel("Irradiación")
axes[0].set_ylabel("Potencia AC")

# Planta 2
sns.scatterplot(
    data=df_full[df_full["PLANT"] == 2],
    x="IRRADIATION",
    y="AC_POWER",
    hue="MODULE_TEMPERATURE",
    palette="inferno",
    alpha=0.6,
    edgecolor=None,
    ax=axes[1],
)

axes[1].set_title("Planta 2")
axes[1].set_xlabel("Irradiación")
axes[1].set_ylabel("")

plt.tight_layout()
plt.show()

- Relación monótona en la que a medida que aumenta la cantidad de radiación solar (más sol) se incrementa la generación (más AC) y sube la temperatura.
- Observaciones por planta:
    - **Planta 1** presenta una nube limpia con pocas anomalías. 
    - **Planta 2** muestra una banda densa de outages totales (AC=0 con sol alto) y un AC muy por debajo de lo esperado dado el nivel de irradiación.

### 5.5 Análisis por inversor

In [21]:
# Filtramos horas de producción (IRRADIATION > 0) para evitar sesgo de valores nocturnos
df_daytime = df_full[df_full["IRRADIATION"] > 0].copy()
# El ratio AC/DC normalizado es el indicador más directo de la salud de un inversor
# Un ratio bajo señala degradación o fallo
df_daytime["DC_AC_RATIO"] = df_daytime["AC_POWER"] / df_daytime["DC_POWER_NORMALIZED"].replace(
    0, np.nan
)

fig, axes = plt.subplots(2, 2, figsize=(22, 12))

plot_cfg = [
    (
        "AC_POWER",
        "AC_POWER por inversor (solo en horas de producción)",
        "Potencia AC (W)",
    ),
    (
        "DC_AC_RATIO",
        "Ratio DC/AC por inversor (solo en horas de producción)",
        "AC/DC normalizado",
    ),
]

for col_idx, (col, title, ylabel) in enumerate(plot_cfg):
    for row_idx, plant_id in enumerate([1, 2]):
        ax = axes[row_idx][col_idx]
        subset = df_daytime[df_daytime["PLANT"] == plant_id]
        order = subset.groupby("SOURCE_KEY")[col].median().sort_values().index
        sns.boxplot(
            data=subset,
            x="SOURCE_KEY",
            y=col,
            order=order,
            ax=ax,
            palette="Blues_r",
            hue="SOURCE_KEY",
            legend=False,
        )

        if col == "DC_AC_RATIO":
            ax.axhline(
                1.0,
                color="red",
                linestyle="--",
                linewidth=1,
                alpha=0.7,
                label="Ratio ideal (1.0)",
            )
            ax.legend(fontsize=9)

        ax.set_title(f"Planta {plant_id} — {title}", fontsize=12)
        ax.set_xlabel("Inversor", fontsize=10)
        ax.set_ylabel(ylabel, fontsize=10)
        ax.tick_params(axis="x", rotation=45)

plt.tight_layout()
plt.show()

- Planta 1 presenta inversores homogéneos (medianas AC 500–600W, ratio DC/AC estable en, más o menos, 0,98).
- Planta 2 muestra heterogeneidad marcada: medianas varían entre 200W y 450W, sugiriendo degradación diferente de la planta. Los outliers del ratio >1.0 son físicamente imposibles y se atribuyen a ruido de medición. Esta heterogeneidad refuerza la evaluación posterior de modelos "per-plant" frente al modelo único.

In [22]:
# Las variables temporales, como la hora del dí­a o el mes del año, son cí­clicas.
# Los modelos de ML las interpretan de forma secuencial y lineal.
# https://www.tensorflow.org/tutorials/structured_data/time_series#time

# Transformación seno/coseno de las fechas para la red
df_full["MINUTE_OF_DAY"] = df_full["DATE_TIME"].dt.hour * 60 + df_full["DATE_TIME"].dt.minute
df_full["TIME_SIN"] = np.sin(2 * np.pi * df_full["MINUTE_OF_DAY"] / (24 * 60))
df_full["TIME_COS"] = np.cos(2 * np.pi * df_full["MINUTE_OF_DAY"] / (24 * 60))

### 5.7 Huecos temporales

In [23]:
# Análisis de huecos temporales por inversor
# Cada par de registros consecutivos del mismo inversor debería estar separado exactamente (15 minutos)
# Los huecos más largos indican caídas del datalogger o inversores fuera de servicio
# !! Importante: create_sequences descartará las secuencias que crucen estos huecos

gap_records = []

for source_key in df_full["SOURCE_KEY"].unique():
    inv_df = df_full[df_full["SOURCE_KEY"] == source_key].sort_values("DATE_TIME")
    diffs = inv_df["DATE_TIME"].diff().dropna()
    gaps = diffs[diffs > pd.Timedelta(minutes=15)]
    gap_records.append(
        {
            "SOURCE_KEY": source_key,
            "PLANT": df_full[df_full["SOURCE_KEY"] == source_key]["PLANT"].iloc[0],
            "total_records": len(inv_df),
            "n_gaps": len(gaps),
            "max_gap_min": int(gaps.max().total_seconds() / 60) if len(gaps) > 0 else 0,
            "total_missing_intervals": int(
                (gaps - pd.Timedelta(minutes=15)).sum().total_seconds() / 60 / 15
            )
            if len(gaps) > 0
            else 0,
        }
    )

df_gaps = pd.DataFrame(gap_records).sort_values(["PLANT", "n_gaps"], ascending=[True, False])
print(df_gaps.to_string(index=False))

In [24]:
# Investigación de los inversores con gap masivo (Planta 2)
# Los 4 inversores con solo ~2355 registros vs ~3259 del resto tienen un hueco de ~9 días
# Este tipo de ausencia no la captura flag_outage (no hay filas que filtrar, los registros simplemente no existen)
# create_sequences lo maneja correctamente al detectar la discontinuidad temporal

ANOMALOUS_INVERTERS = [
    "IQ2d7wF4YD8zU1Q",
    "NgDl19wMapZy17u",
    "mqwcsP2rE7J0TFp",
    "xMbIugepa2P7lBB",
]

for source_key in ANOMALOUS_INVERTERS:
    inv_df = df_full[df_full["SOURCE_KEY"] == source_key].sort_values("DATE_TIME")
    diffs = inv_df["DATE_TIME"].diff()
    gap_idx = diffs[diffs > pd.Timedelta(minutes=15)].index

    for idx in gap_idx:
        pos = inv_df.index.get_loc(idx)
        gap_start = inv_df["DATE_TIME"].iloc[pos - 1]
        gap_end = inv_df["DATE_TIME"].iloc[pos]
        duration = gap_end - gap_start
        print(
            f"{source_key}: gap desde {gap_start} hasta {gap_end}  ({int(duration.total_seconds() / 3600)} horas)"
        )

## 6. División del dataset

In [25]:
# División de los datos

# Nota: este dataset no es una simple línea temporal, es una línea múltiple donde cada inversor tiene su propia línea temporal transcurriendo simultáneamente
# Al ordenar el dataset como Planta, Inversor y Fecha aparecen apilados uno detrás del otro

# Procedimiento:
# 1. Extrae una lista limpia de los días que existen en el dataset (día 1, día 2... día 34).
# 2. Calcula qué día marca exactamente el 60% del tiempo transcurrido (por ejemplo, el día 20).
# 3. Usa TODOS los registros de todos los inversores, pero solo hasta ese día 20.

# Obtenemos las fechas únicas ordenadas
unique_dates = df_full["DATE_TIME"].dt.date.sort_values().unique()

# Calcular índices de corte temporal (60% Train, 20% Val, 20% Prod)
idx_train = int(len(unique_dates) * 0.60)
idx_val = int(len(unique_dates) * 0.80)

train_end_date = unique_dates[idx_train]
val_end_date = unique_dates[idx_val]

# Separar respetando el tiempo para TODOS los inversores
df_train = df_full[df_full["DATE_TIME"].dt.date <= train_end_date].copy()
df_val = df_full[
    (df_full["DATE_TIME"].dt.date > train_end_date) & (df_full["DATE_TIME"].dt.date <= val_end_date)
].copy()
df_prod = df_full[df_full["DATE_TIME"].dt.date > val_end_date].copy()

n_days_train = idx_train + 1
n_days_val = idx_val - idx_train
n_days_prod = len(unique_dates) - idx_val - 1

print(
    f"Rango temporal total : {unique_dates[0]} --> {unique_dates[-1]}  ({len(unique_dates)} días)"
)
print("\nEstructura de la división:")
print(
    f"  - Train: {unique_dates[0]} --> {train_end_date} | {len(df_train):>6} registros | {n_days_train} días"
)
print(
    f"  - Validación: {unique_dates[idx_train + 1]} --> {val_end_date} | {len(df_val):>6} registros | {n_days_val} días"
)
print(
    f"  - Producción: {unique_dates[idx_val + 1]} --> {unique_dates[-1]} | {len(df_prod):>6} registros | {n_days_prod} días"
)

## 7. Preprocesamiento y funciones núcleo

> Preparamos los datos "sanos" para entrenamiento y definimos **todas las funciones reutilizables** del pipeline. Estas funciones se usan tanto en el entrenamiento principal, como en la evaluación en producción y en el estudio de relevancia *per-plant*.

### 7.1 Corrección de escala y máscara de registros sanos

> `DC_POWER` de la planta 1 viene en una escala 10 veces superior (dividimos entre 10). `make_healthy_mask` marca los registros sin fallos conocidos (outage con sol y arrastre de `DAILY_YIELD`), que son los únicos que queremos usar para enseñar al modelo "cómo es un funcionamiento correcto".

In [26]:
for df in [df_train, df_val, df_prod]:
    df.loc[df["PLANT"] == 1, "DC_POWER"] = df.loc[df["PLANT"] == 1, "DC_POWER"] / 10

sun_umbral = 0.1


def make_healthy_mask(df):
    # Hay sol pero el inversor no produce energía (outage o fallo de inversor)
    flag_outage = (df["IRRADIATION"] > sun_umbral) & (df["AC_POWER"] == 0)
    # El contador DAILY_YIELD no se reinicia a cero a medianoche (arrastre del día anterior)
    flag_yield = (
        (df["IRRADIATION"] == 0) & (df["DATE_TIME"].dt.hour < 5) & (df["DAILY_YIELD"] > 100)
    )
    return ~(flag_outage | flag_yield)


df_train_healthy = df_train[make_healthy_mask(df_train)].copy()
df_val_healthy = df_val[make_healthy_mask(df_val)].copy()

n_removed_train = len(df_train) - len(df_train_healthy)
n_removed_val = len(df_val) - len(df_val_healthy)

print(f"\nTrain total: {len(df_train):>6}")
print(f" - Sanos: {len(df_train_healthy):>6}")
print(f" - Eliminados: {n_removed_train:>5} ({n_removed_train / len(df_train) * 100:.1f}%)")
print(f"\nVal total: {len(df_val):>6}")
print(f" - Sanos: {len(df_val_healthy):>6}")
print(f" - Eliminados: {n_removed_val:>5} ({n_removed_val / len(df_val) * 100:.1f}%)")

## 7.2 Features y ventana temporal

> El modelo no ve filas sueltas sino **ventanas contiguas de 32 pasos** (aproximadamente 8 horas, 15 min entre lecturas). Cada ventana resume el comportamiento reciente del inversor. `TIME_SIN` / `TIME_COS` codifican la hora del día como coordenadas cíclicas.

In [27]:
features = [
    "DC_POWER",
    "AC_POWER",
    "AMBIENT_TEMPERATURE",
    "MODULE_TEMPERATURE",
    "IRRADIATION",
    "TIME_SIN",
    "TIME_COS",
]
# Más o menos 8 horas de contexto
TIME_STEPS = 32

### 7.3 Generación de ventanas deslizantes

In [28]:
# Genera ventanas contiguas por inversor validando continuidad temporal (15 min/paso).
#  - Si filter_night=True, descarta ventanas completamente nocturnas (max IRRADIATION < sun_umbral).
#  - Si anomaly_mask se pasa, devuelve también las etiquetas por ventana (regla "última fila").
def create_sequences(df_original, scaled_data, time_steps, anomaly_mask=None, filter_night=False):
    n_features = scaled_data.shape[1]
    inverters = df_original["SOURCE_KEY"].values
    times = df_original["DATE_TIME"].values
    irr_values = df_original["IRRADIATION"].values
    expected_minutes = (time_steps - 1) * 15

    sequences: list = []
    window_labels: list = []
    n_night_discarded = 0

    for source_key in pd.unique(inverters):
        positions = np.where(inverters == source_key)[0]
        if len(positions) < time_steps:
            continue

        inv_scaled = scaled_data[positions]
        inv_times = times[positions]

        windows = np.lib.stride_tricks.sliding_window_view(
            inv_scaled, window_shape=time_steps, axis=0
        ).transpose(0, 2, 1)

        n_seq = windows.shape[0]
        time_diffs = (
            (inv_times[time_steps - 1 :] - inv_times[:n_seq])
            .astype("timedelta64[m]")
            .astype(np.int64)
        )
        valid = time_diffs == expected_minutes

        if filter_night:
            inv_irr = irr_values[positions]
            irr_windows = np.lib.stride_tricks.sliding_window_view(inv_irr, window_shape=time_steps)
            daytime = irr_windows.max(axis=1) >= sun_umbral
            n_night_discarded += int(np.sum(valid & ~daytime))
            valid = valid & daytime

        sequences.append(windows[valid])

        if anomaly_mask is not None:
            inv_anom = np.asarray(anomaly_mask)[positions]
            anom_windows = np.lib.stride_tricks.sliding_window_view(
                inv_anom, window_shape=time_steps
            )
            # Regla de la "última fila"
            # La ventana se etiqueta por el estado del instante final
            window_labels.append(anom_windows[:, -1][valid])

    if filter_night and n_night_discarded > 0:
        print(f"[create_sequences] Filtro nocturno: {n_night_discarded} ventanas descartadas")

    if not sequences:
        empty_x = np.empty((0, time_steps, n_features))
        if anomaly_mask is not None:
            return empty_x, np.empty(0, dtype=bool)
        return empty_x

    X = np.concatenate(sequences, axis=0)
    if anomaly_mask is not None:
        y = np.concatenate(window_labels, axis=0)
        return X, y
    return X

### 7.4 Arquitectura del Autoencoder LSTM
> Encoder-Decoder con cuello de botella de 8 unidades. El encoder comprime la ventana hasta 8 números, el decoder la reconstruye. Cuanto más diferente es la reconstrucción del original, más "anómala" es la ventana. LayerNorm + Dropout 0.3 estabilizan el entrenamiento y reducen sobreajuste.

In [29]:
def build_model(time_steps, n_features):
    m = Sequential()
    m.add(Input(shape=(time_steps, n_features)))
    m.add(Bidirectional(LSTM(64, return_sequences=True)))
    m.add(LayerNormalization())
    m.add(Dropout(0.3))
    m.add(LSTM(8, return_sequences=False))
    m.add(RepeatVector(time_steps))
    m.add(LSTM(8, return_sequences=True))
    m.add(LayerNormalization())
    m.add(Dropout(0.3))
    m.add(LSTM(64, return_sequences=True))
    m.add(LayerNormalization())
    m.add(Dropout(0.3))
    m.add(TimeDistributed(Dense(n_features)))
    m.compile(optimizer=Adam(learning_rate=1e-3, clipnorm=1.0), loss="mae")
    return m


def build_mlp_head(bottleneck_dim):
    # Cabezal MLP que predice AC_POWER (escalado) desde el cuello de botella del AE.
    h = Sequential(
        [
            Input(shape=(bottleneck_dim,)),
            Dense(64, activation="relu"),
            Dense(32, activation="relu"),
            Dense(1),
        ]
    )
    h.compile(optimizer=Adam(learning_rate=1e-3), loss="mae")
    return h


def make_encoder(ae_model, bottleneck_layer_idx=3):
    # Extrae encoder hasta el cuello de botella reutilizando los pesos ya entrenados.
    return Model(inputs=ae_model.inputs, outputs=ae_model.layers[bottleneck_layer_idx].output)

### 7.5 Entrenamiento parametrizable

> Una sola función encapsula escalado + ventanas + fit. Se parametriza la semilla para reproducibilidad y comparativas multi-seed.

In [30]:
def train_model(
    df_train_healthy,
    df_val_healthy,
    features_list,
    time_steps,
    seed=42,
    epochs=100,
    batch_size=128,
    verbose=0,
):
    set_seeds(seed)

    sc = MinMaxScaler()
    sc.fit(df_train_healthy[features_list])
    tr_scaled = sc.transform(df_train_healthy[features_list])
    va_scaled = sc.transform(df_val_healthy[features_list])

    X_tr = create_sequences(df_train_healthy, tr_scaled, time_steps)
    X_va = create_sequences(df_val_healthy, va_scaled, time_steps)

    m = build_model(time_steps, len(features_list))

    cb_stop = EarlyStopping(
        monitor="val_loss",
        patience=10,
        restore_best_weights=True,
        mode="min",
        verbose=verbose,
    )
    cb_lr = ReduceLROnPlateau(
        monitor="val_loss", factor=0.5, patience=4, min_lr=1e-5, verbose=verbose
    )

    hist = m.fit(
        X_tr,
        X_tr,
        epochs=epochs,
        batch_size=batch_size,
        validation_data=(X_va, X_va),
        callbacks=[cb_stop, cb_lr],
        shuffle=True,
        verbose=verbose,
    )

    # Cabezal MLP sobre el cuello de botella del encoder
    # Objetivo --> AC_POWER escalado del último timestep de la ventana
    encoder = make_encoder(m, bottleneck_layer_idx=3)
    ac_idx = features_list.index("AC_POWER")
    Z_tr = encoder.predict(X_tr, verbose=0)
    Z_va = encoder.predict(X_va, verbose=0)
    y_tr = X_tr[:, -1, ac_idx]
    y_va = X_va[:, -1, ac_idx]

    head = build_mlp_head(Z_tr.shape[1])
    cb_stop_h = EarlyStopping(
        monitor="val_loss", patience=10, restore_best_weights=True, mode="min", verbose=verbose
    )
    cb_lr_h = ReduceLROnPlateau(
        monitor="val_loss", factor=0.5, patience=4, min_lr=1e-5, verbose=verbose
    )
    head.fit(
        Z_tr,
        y_tr,
        validation_data=(Z_va, y_va),
        epochs=epochs,
        batch_size=batch_size,
        callbacks=[cb_stop_h, cb_lr_h],
        shuffle=True,
        verbose=verbose,
    )

    return m, encoder, head, sc, hist

### 7.6 Scoring, umbral y evaluación

> El modelo produce un error de reconstrucción por cada celda `(ventana, paso, feature)`. Para decidir si una ventana es anómala hay que resumir esos errores en un único número (**scoring**) y compararlo con un umbral. Probamos cuatro estrategias de scoring y seleccionamos la que mejor separa sanas de anómalas: `D_p95_power` (percentil 95 del error sobre `DC_POWER`/`AC_POWER`).

In [31]:
def _compute_forecast_errors(model_fit, encoder_fit, head_fit, X, features_list):
    # Error del MLP head (escalado, scalar por ventana).
    if encoder_fit is None or head_fit is None:
        return None
    ac_idx = features_list.index("AC_POWER")
    Z = encoder_fit.predict(X, verbose=0)
    y_pred = head_fit.predict(Z, verbose=0).reshape(-1)
    y_true = X[:, -1, ac_idx]
    return np.abs(y_true - y_pred)


def _score_windows(abs_errors, features_list, scoring, forecast_errors=None):
    p_idx = [features_list.index("DC_POWER"), features_list.index("AC_POWER")]
    if scoring == "A_mae_global":
        return np.mean(abs_errors, axis=(1, 2))
    if scoring == "B_mae_power":
        return np.mean(abs_errors[:, :, p_idx], axis=(1, 2))
    if scoring == "C_p95_window":
        return np.percentile(np.mean(abs_errors, axis=2), 95, axis=1)
    if scoring == "D_p95_power":
        return np.percentile(np.mean(abs_errors[:, :, p_idx], axis=2), 95, axis=1)
    if scoring == "E_mlp_only":
        if forecast_errors is None:
            raise ValueError("E_mlp_only requiere forecast_errors")
        return forecast_errors
    if scoring == "F_hybrid":
        # Para la suma ya tenemos ambos términos escalados y con magnitudes comparables
        if forecast_errors is None:
            raise ValueError("F_hybrid requiere forecast_errors")
        recon = np.percentile(np.mean(abs_errors[:, :, p_idx], axis=2), 95, axis=1)
        return recon + forecast_errors
    raise ValueError(f"scoring desconocido: {scoring}")


# Calcula umbral F1-óptimo sobre df_val etiquetado con flag_outage
def compute_threshold(
    model_fit,
    scaler_fit,
    df_val_labeled,
    features_list,
    time_steps,
    scoring="D_p95_power",
    encoder_fit=None,
    head_fit=None,
):
    val_scaled = scaler_fit.transform(df_val_labeled[features_list])
    mask_out = (
        (df_val_labeled["IRRADIATION"] > sun_umbral) & (df_val_labeled["AC_POWER"] == 0)
    ).values

    X_va, y_out = create_sequences(df_val_labeled, val_scaled, time_steps, anomaly_mask=mask_out)
    y_out = np.asarray(y_out, dtype=bool)

    X_va_pred = model_fit.predict(X_va, verbose=0)
    forecast_err = _compute_forecast_errors(model_fit, encoder_fit, head_fit, X_va, features_list)
    scores = _score_windows(np.abs(X_va_pred - X_va), features_list, scoring, forecast_err)

    prec_c, rec_c, thr_c = precision_recall_curve(y_out, scores)
    f1_c = 2 * prec_c[:-1] * rec_c[:-1] / (prec_c[:-1] + rec_c[:-1] + 1e-12)
    best = int(np.argmax(f1_c))
    return float(thr_c[best]), scoring


# Evalúa modelo + umbral sobre df_val y devolvemos una estructura con las métricas
def evaluate(
    model_fit,
    scaler_fit,
    threshold_value,
    df_val_labeled,
    features_list,
    time_steps,
    scoring="D_p95_power",
    encoder_fit=None,
    head_fit=None,
):
    val_scaled = scaler_fit.transform(df_val_labeled[features_list])
    mask_out = (
        (df_val_labeled["IRRADIATION"] > sun_umbral) & (df_val_labeled["AC_POWER"] == 0)
    ).values

    X_va, y_out = create_sequences(df_val_labeled, val_scaled, time_steps, anomaly_mask=mask_out)
    y_out = np.asarray(y_out, dtype=bool)

    X_va_pred = model_fit.predict(X_va, verbose=0)
    abs_errors = np.abs(X_va_pred - X_va)
    forecast_err = _compute_forecast_errors(model_fit, encoder_fit, head_fit, X_va, features_list)
    scores = _score_windows(abs_errors, features_list, scoring, forecast_err)

    y_pred = scores > threshold_value
    tp = int(np.sum(y_pred & y_out))
    fp = int(np.sum(y_pred & ~y_out))
    fn = int(np.sum(~y_pred & y_out))
    tn = int(np.sum(~y_pred & ~y_out))
    prec = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    rec = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    f1 = 2 * prec * rec / (prec + rec) if (prec + rec) > 0 else 0.0

    roc = float(roc_auc_score(y_out, scores)) if y_out.any() and (~y_out).any() else float("nan")
    prec_c, rec_c, _ = precision_recall_curve(y_out, scores)
    pr = float(auc(rec_c, prec_c))

    return {
        "roc_auc": roc,
        "pr_auc": pr,
        "precision": prec,
        "recall": rec,
        "f1": f1,
        "tp": tp,
        "fp": fp,
        "tn": tn,
        "fn": fn,
        "threshold": float(threshold_value),
        "scoring": scoring,
        "n_windows": int(len(X_va)),
        "n_positives": int(y_out.sum()),
        "scores": scores,
        "y_outage": y_out,
        "abs_errors": abs_errors,
        "X": X_va,
    }

## 8. Entrenamiento del modelo

> Entrenamos la arquitectura F2B (front to back) con la función `train_model` sobre el split combinado planta 1 + planta 2 con semilla fija 42.

In [32]:
model, encoder, mlp_head, scaler, history = train_model(
    df_train_healthy,
    df_val_healthy,
    features,
    TIME_STEPS,
    seed=42,
    verbose=1,
)

model.summary()
mlp_head.summary()
plot_model(model, show_shapes=True, show_layer_names=True, dpi=150)

In [33]:
plt.figure(figsize=(10, 6))
plt.plot(history.history["loss"], label="Pérdida de entrenamiento (Train Loss)")
plt.plot(history.history["val_loss"], label="Pérdida de validación (Val Loss)")
plt.title("Curva de aprendizaje del LSTM Autoencoder", fontsize=14)
plt.xlabel("Épocas", fontsize=12)
plt.ylabel("Error Absoluto Medio (MAE)", fontsize=12)
plt.legend()
plt.tight_layout()
plt.show()

## 9. Validación del modelo

Evaluamos el modelo contra las anomalías reales que eliminamos en limpieza. De las dos máscaras:

- **`flag_outage`** (sol pero sin AC): **detectable sencillamente**.
- **`flag_yield`** (DAILY_YIELD sin reset): **fuera de scope** (la feature no está en el vector de entrada).

Las métricas **ROC-AUC** y **PR-AUC** son independientes del umbral. El umbral operativo se fija al punto **F1-óptimo** de la curva Precisión-Recall.

### 9.1 Comparativa de estrategias de scoring

> El modelo produce un tensor de errores por ventana de forma `(32, 7)`. Para decidir si una ventana es anómala hay que resumirlo en un único número.

Comparamos cuatro estrategias:

- **A** `mae_global`: promedio sobre todas las celdas.
- **B** `mae_power`: promedio solo sobre `DC_POWER` y `AC_POWER`.
- **C** `p95_window`: percentil 95 temporal sobre todas las celdas.
- **D** `p95_power`: percentil 95 temporal sobre `DC_POWER` y `AC_POWER`.

Seleccionamos la estrategia con mejor PR-AUC.

In [34]:
# Evaluamos las 6 estrategias con umbral dummy (roc/pr son independientes del umbral)
print("=" * 70)
print("Comparativa de estrategias de scoring")
print("=" * 70)

best_strategy, best_pr_auc = None, 0.0
strategy_scores = {}

for name in [
    "A_mae_global",
    "B_mae_power",
    "C_p95_window",
    "D_p95_power",
    "E_mlp_only",
    "F_hybrid",
]:
    ev = evaluate(
        model,
        scaler,
        0.0,
        df_val,
        features,
        TIME_STEPS,
        scoring=name,
        encoder_fit=encoder,
        head_fit=mlp_head,
    )
    strategy_scores[name] = ev
    print(f"\n[{name}] ROC-AUC={ev['roc_auc']:.4f} | PR-AUC={ev['pr_auc']:.4f}")
    if ev["pr_auc"] > best_pr_auc:
        best_pr_auc = ev["pr_auc"]
        best_strategy = name

print(f"\n{'=' * 70}")
print(f"Mejor estrategia por PR-AUC: {best_strategy} ({best_pr_auc:.4f})")
print(f"{'=' * 70}")

### 9.2 Umbral F1-óptimo y métricas finales

In [35]:
# Umbral F1-óptimo sobre la estrategia ganadora
optimal_threshold, _ = compute_threshold(
    model,
    scaler,
    df_val,
    features,
    TIME_STEPS,
    scoring=best_strategy,
    encoder_fit=encoder,
    head_fit=mlp_head,
)

# Evaluación final con el umbral operativo
eval_val = evaluate(
    model,
    scaler,
    optimal_threshold,
    df_val,
    features,
    TIME_STEPS,
    scoring=best_strategy,
    encoder_fit=encoder,
    head_fit=mlp_head,
)

# Variables para usar en todas las celdas posteriores
selected_scores = eval_val["scores"]
y_outage = eval_val["y_outage"]
roc_auc_val = eval_val["roc_auc"]
pr_auc_val = eval_val["pr_auc"]
metrics_optimal = {k: eval_val[k] for k in ["precision", "recall", "f1", "tp", "fp", "tn", "fn"]}
umbral = optimal_threshold
power_idx = [features.index("DC_POWER"), features.index("AC_POWER")]

# Curva PR para plots
prec_curve, rec_curve, thresholds_pr = precision_recall_curve(y_outage, selected_scores)
f1_curve = 2 * prec_curve[:-1] * rec_curve[:-1] / (prec_curve[:-1] + rec_curve[:-1] + 1e-12)
best_idx = int(np.argmax(f1_curve))

print(f"\nEstrategia: {best_strategy}")
print(f" - Umbral F1-óptimo: {optimal_threshold:.4f}")
print(f" - ROC-AUC: {roc_auc_val:.4f} | PR-AUC: {pr_auc_val:.4f}")
print(
    f" - Precisión: {metrics_optimal['precision']:.3f} | Recall: {metrics_optimal['recall']:.3f} | F1: {metrics_optimal['f1']:.3f}"
)
print(
    f" - TP={metrics_optimal['tp']} FP={metrics_optimal['fp']} TN={metrics_optimal['tn']} FN={metrics_optimal['fn']}"
)

### 9.3 Umbrales estadísticos de referencia (μ+3σ, p95)

Adicional al umbral F1-óptimo (que requiere etiquetas), a modo de comparativa, calculamos dos umbrales puramente estadísticos calculados sobre la **validación sana** (regla empírica 99.7% y percentil 95). Son útiles como umbrales de respaldo cuando no se dispone de anomalías etiquetadas.

In [36]:
# MAE global por ventana sobre validación sana (scoring A equivalente)
val_scaled_healthy = scaler.transform(df_val_healthy[features])
X_val_healthy = create_sequences(df_val_healthy, val_scaled_healthy, TIME_STEPS)
val_mae_loss = np.mean(np.abs(model.predict(X_val_healthy, verbose=0) - X_val_healthy), axis=(1, 2))

mean_val = float(np.mean(val_mae_loss))
std_val = float(np.std(val_mae_loss))
robust_threshold = float(mean_val + 3 * std_val)
percentile_threshold = float(np.percentile(val_mae_loss, 95))

print("Error validación sana (scoring A_mae_global):")
print(f" - Mean: {mean_val:.4f} | std: {std_val:.4f}")
print(f" - Umbral robusto μ+3σ: {robust_threshold:.4f}")
print(f" - Umbral percentil 95: {percentile_threshold:.4f}")

### 9.4 Distribución del error (sanas vs anómalas)

In [37]:
healthy_errors = selected_scores[~y_outage]
anomalous_errors = selected_scores[y_outage]
upper = float(max(selected_scores.max(), optimal_threshold * 2.0))
bins = np.linspace(0, upper, 100)

plt.figure(figsize=(11, 6))
plt.hist(
    healthy_errors,
    bins=bins,
    color="seagreen",
    alpha=0.55,
    label=f"Sanas (n={len(healthy_errors)})",
    density=True,
)
plt.hist(
    anomalous_errors,
    bins=bins,
    color="crimson",
    alpha=0.55,
    label=f"Anómalas (n={len(anomalous_errors)})",
    density=True,
)
plt.axvline(
    optimal_threshold,
    color="darkorange",
    linestyle="--",
    linewidth=2,
    label=f"Umbral F1-óptimo {optimal_threshold:.4f}",
)
plt.title(f"Error de reconstrucción — {best_strategy} (flag_outage df_val)", fontsize=14)
plt.xlabel("Score de anomalía", fontsize=12)
plt.ylabel("Densidad", fontsize=12)
plt.legend()
plt.tight_layout()
plt.show()

### 9.5 Curva Precisión-Recall

In [38]:
plt.figure(figsize=(9, 6))
plt.plot(
    rec_curve,
    prec_curve,
    color="steelblue",
    linewidth=2,
    label=f"PR curve (AUC={pr_auc_val:.4f})",
)
plt.scatter(
    [rec_curve[best_idx]],
    [prec_curve[best_idx]],
    color="darkorange",
    s=120,
    zorder=5,
    label=f"F1-óptimo (P={prec_curve[best_idx]:.2f}, R={rec_curve[best_idx]:.2f})",
)
plt.xlabel("Recall", fontsize=12)
plt.ylabel("Precisión", fontsize=12)
plt.title("Curva Precisión-Recall sobre flag_outage (df_val)", fontsize=14)
plt.legend(fontsize=11)
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

### 9.6 Diagnósticos complementarios

**MAE por feature:** verifica la hipótesis de que `DC_POWER` y `AC_POWER` dominan el error de reconstrucción (justifica scoring D sobre potencias).

- **`DC_AC_RATIO`:** distribución del ratio diurno.
- **Latencia:** tiempo medio de inferencia sobre una ventana (interesante ya que el entorno de producción será un entorno edge).

In [39]:
per_feature_mae = np.mean(eval_val["abs_errors"], axis=(0, 1))
print("MAE medio por feature sobre df_val completo:")
for nm, err in zip(features, per_feature_mae):
    print(f" - {nm:<22} {err:.4f}")

In [40]:
dr = df_full[(df_full["IRRADIATION"] > 0.1) & (df_full["DC_POWER_NORMALIZED"] > 0)].copy()
dr["DC_AC_RATIO"] = dr["AC_POWER"] / dr["DC_POWER_NORMALIZED"]
print(dr["DC_AC_RATIO"].describe())

plt.figure(figsize=(10, 5))
sns.histplot(dr["DC_AC_RATIO"], bins=100, color="steelblue", kde=True)
plt.title("Distribución de DC_AC_RATIO (horas diurnas, DC_POWER > 0)", fontsize=14)
plt.xlabel("AC_POWER / DC_POWER", fontsize=12)
plt.ylabel("Frecuencia", fontsize=12)
plt.tight_layout()
plt.show()

In [ ]:
# Descartar la primera llamada (compilación grafo + cuDNN)
_ = model.predict(eval_val["X"][:1], verbose=0)

n_reps = 100
start = time.perf_counter()
for _ in range(n_reps):
    _ = model.predict(eval_val["X"][:1], verbose=0)
elapsed = time.perf_counter() - start
latency_ms = (elapsed / n_reps) * 1000
print(f"Latencia media por ventana: {latency_ms:.2f} ms ({n_reps} repeticiones)")

## 10. Análisis de modelos baseline

- **Isolation Forest**: mismo split sano y mismas features.
- **Regresión lineal**: `IRRADIATION` + `MODULE_TEMPERATURE` --> `AC_POWER`.

### 10.1 Isolation forest

Entrenado sobre las mismas features escaladas y el mismo split sano. Evaluado con `decision_function` (score continuo) sobre **df_val** completo para calcular **ROC-AUC** y **PR-AUC** comparables al LSTM-AE.

In [ ]:
iso_forest = IsolationForest(
    n_estimators=200,
    contamination="auto",
    random_state=42,
    n_jobs=-1,
)
train_scaled = scaler.transform(df_train_healthy[features])
iso_forest.fit(train_scaled)

val_full_scaled = scaler.transform(df_val[features])

if_scores_raw = iso_forest.decision_function(val_full_scaled)
# Convertimos que sea compatible con nuestras métricas (más alto = más anómalo)
if_scores = -if_scores_raw

# Etiquetas por fila (misma definición que flag_outage)
mask_outage_rows = ((df_val["IRRADIATION"] > sun_umbral) & (df_val["AC_POWER"] == 0)).values

roc_if = roc_auc_score(mask_outage_rows, if_scores)
prec_if, rec_if, thr_if = precision_recall_curve(mask_outage_rows, if_scores)
pr_if = auc(rec_if, prec_if)

f1_if = 2 * prec_if[:-1] * rec_if[:-1] / (prec_if[:-1] + rec_if[:-1] + 1e-12)
best_if_idx = np.argmax(f1_if)
if_threshold = float(thr_if[best_if_idx])

print("Isolation Forest:")
print(f" - ROC-AUC: {roc_if:.4f} | PR-AUC: {pr_if:.4f}")
print(
    f" - F1-óptimo: {f1_if[best_if_idx]:.4f} (P={prec_if[best_if_idx]:.3f}, R={rec_if[best_if_idx]:.3f})"
)
print(f" - Umbral F1-óptimo: {if_threshold:.4f}")
print(f" - Filas evaluadas: {len(if_scores)} | Anomalías (flag_outage): {mask_outage_rows.sum()}")

### 10.2 Regresión lineal

> Modelo trivial

Predice `AC_POWER` a partir de `IRRADIATION` y `MODULE_TEMPERATURE` (ajuste sin intercept, entrenado sobre split sano). Un resultado muy negativo indica que la planta produce menos de lo esperado (anomalía). Evaluación **por fila**.

In [ ]:
lr_features = ["IRRADIATION", "MODULE_TEMPERATURE"]

X_lr_train = df_train_healthy[lr_features].values
y_lr_train = df_train_healthy["AC_POWER"].values

lr_model = LinearRegression(fit_intercept=False)
lr_model.fit(X_lr_train, y_lr_train)

print(
    f"Coeficientes LR: IRRADIATION={lr_model.coef_[0]:.2f}, MODULE_TEMPERATURE={lr_model.coef_[1]:.2f}"
)
print(f"R^2 sobre train sano: {lr_model.score(X_lr_train, y_lr_train):.4f}")

# Predicción sobre df_val completo
X_lr_val = df_val[lr_features].values
y_lr_val = df_val["AC_POWER"].values

lr_pred = lr_model.predict(X_lr_val)
lr_residuals = y_lr_val - lr_pred
lr_scores = -lr_residuals

roc_lr = roc_auc_score(mask_outage_rows, lr_scores)
prec_lr, rec_lr, thr_lr = precision_recall_curve(mask_outage_rows, lr_scores)
pr_lr = auc(rec_lr, prec_lr)

f1_lr = 2 * prec_lr[:-1] * rec_lr[:-1] / (prec_lr[:-1] + rec_lr[:-1] + 1e-12)
best_lr_idx = np.argmax(f1_lr)
lr_threshold = float(thr_lr[best_lr_idx])

print("\nRegresión Lineal Residual:")
print(f" - ROC-AUC: {roc_lr:.4f}  |  PR-AUC: {pr_lr:.4f}")
print(
    f" - F1-óptimo: {f1_lr[best_lr_idx]:.4f}  (P={prec_lr[best_lr_idx]:.3f}, R={rec_lr[best_lr_idx]:.3f})"
)
print(f" - Umbral F1-óptimo: {lr_threshold:.4f}")

### 10.3 Tabla comparativa final

Los baselines (RL + IF) superan ampliamente al LSTM-AE en detección aislada de `flag_outage`. Esto es esperable porque `flag_outage` es una anomalía puntual (detectable en un solo instante sin contexto temporal). Los baselines evalúan por fila mientras que el LSTM-AE evalúa por ventana de 32 pasos.
 
> **Nota:** la diferencia de granularidad (fila vs ventana de 32 pasos) favorece estructuralmente a los baselines para
anomalías puntuales. El LSTM-AE aporta valor al pipeline por su error descompuesto por feature y timestep, que enriquece la explicabilidad final.

In [ ]:
print("=" * 80)
print("Comparativa final: LSTM-AE vs BASELINES (flag_outage)")
print("=" * 80)

print(
    f"\n{'Modelo':<25} {'Granularidad':<14} {'ROC-AUC':>8} {'PR-AUC':>8} {'F1':>8} {'Precision':>10} {'Recall':>8}"
)
print("-" * 80)
print(
    f"{'Regresión Lineal':<25} {'por fila':<14} {roc_lr:>8.4f} {pr_lr:>8.4f} {f1_lr[best_lr_idx]:>8.4f} {prec_lr[best_lr_idx]:>10.4f} {rec_lr[best_lr_idx]:>8.4f}"
)
print(
    f"{'Isolation Forest':<25} {'por fila':<14} {roc_if:>8.4f} {pr_if:>8.4f} {f1_if[best_if_idx]:>8.4f} {prec_if[best_if_idx]:>10.4f} {rec_if[best_if_idx]:>8.4f}"
)
print(
    f"{'LSTM-AE (D_p95_power)':<25} {'por ventana':<14} {roc_auc_val:>8.4f} {pr_auc_val:>8.4f} {metrics_optimal['f1']:>8.4f} {metrics_optimal['precision']:>10.4f} {metrics_optimal['recall']:>8.4f}"
)
print("-" * 80)

In [ ]:
# Curvas PR comparativas
fig, axes = plt.subplots(1, 2, figsize=(18, 7))

# Panel izquierdo: las tres curvas juntas
axes[0].plot(
    rec_curve,
    prec_curve,
    color="steelblue",
    linewidth=2,
    label=f"LSTM-AE D_p95_power (PR-AUC={pr_auc_val:.4f})",
)
axes[0].plot(
    rec_if,
    prec_if,
    color="darkorange",
    linewidth=2,
    linestyle="--",
    label=f"Isolation Forest (PR-AUC={pr_if:.4f})",
)
axes[0].plot(
    rec_lr,
    prec_lr,
    color="forestgreen",
    linewidth=2,
    linestyle=":",
    label=f"Regresión Lineal (PR-AUC={pr_lr:.4f})",
)
axes[0].set_xlabel("Recall", fontsize=12)
axes[0].set_ylabel("Precisión", fontsize=12)
axes[0].set_title("Curvas Precision-Recall comparativas (flag_outage)", fontsize=13)
axes[0].legend(fontsize=10)
axes[0].grid(alpha=0.3)

# Panel derecho: distribuciones de scores normalizados para visualizar la separación entre clases por cada modelo
scores_norm = {
    "LSTM-AE": minmax_scale(selected_scores),
    "Isol. Forest": minmax_scale(if_scores),
    "Reg. Lineal": minmax_scale(lr_scores),
}
labels_map = {
    "LSTM-AE": y_outage,
    "Isol. Forest": mask_outage_rows,
    "Reg. Lineal": mask_outage_rows,
}
colors = {
    "LSTM-AE": "steelblue",
    "Isol. Forest": "darkorange",
    "Reg. Lineal": "forestgreen",
}

for i, (name, sc) in enumerate(scores_norm.items()):
    lbl = labels_map[name]
    axes[1].hist(sc[~lbl], bins=80, alpha=0.3, color=colors[name], density=True)
    axes[1].hist(
        sc[lbl],
        bins=80,
        alpha=0.6,
        color=colors[name],
        density=True,
        label=f"{name} (anómalas)",
        histtype="step",
        linewidth=2,
    )

axes[1].set_xlabel("Score normalizado [0, 1]", fontsize=12)
axes[1].set_ylabel("Densidad", fontsize=12)
axes[1].set_title("Separación sanas vs anómalas por modelo (scores normalizados)", fontsize=13)
axes[1].legend(fontsize=9)
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

### 10.4 ¿Por qué los baselines son mejores en este problema?

`flag_outage` se define como `IRRADIATION > 0.1` y `AC_POWER == 0`. Es
una condición **instantánea** detectable en una sola fila:
1. **Regresión lineal (PR-AUC 0.99):** aprende `AC_POWER = f(IRR, T_MOD)`.
   Cuando hay sol y `AC_POWER = 0`, el residuo es enorme. El modelo
   replica casi exactamente la definición de la anomalía.
2. **Isolation Forest (PR-AUC 0.85):** detecta combinaciones raras
   en el espacio de 7 features. Sol alto + potencia cero es una señal fácil de anomalía.
3. **LSTM-AE (PR-AUC 0.27):** reconstruye ventanas de 32 timesteps.
   El outage afecta pocos pasos de la ventana y el error queda oculto. El scoring p95 sobre potencias mitiga esto parcialmente
   (mejora de 0.17 a 0.27), pero no elimina la desventaja estructural.

### Elección del LSTM-AE para el pipeline

A pesar de las métricas inferiores en detección aislada, el LSTM-AE
es el modelo elegido para producción por tres razones:
1. Produce un tensor `(32, 7)` de errores por
   timestep y feature. El módulo explainer utilizaría esta descomposición
   para generar explicaciones contextualizadas ("el error se concentra
   en AC_POWER entre las 14:00 y las 14:30").
2. En un escenario de producción real con
   datos más ricos, las anomalías incluirían degradaciones progresivas,
   patrones estacionales y comportamientos transitorios que
   requieren contexto temporal. Exactamente lo que el LSTM-AE modela
   y los baselines ignoran.
3. El dataset de Kaggle (34 días con anomalías
   naturales limitadas) favorece las anomalías puntuales. Con datos
   industriales reales (meses/años, múltiples tipos de fallo, etc) la
   ventaja de los baselines se reduce.

**Resumen:** los baselines son mejores detectores para este tipo de
anomalía en este dataset, pero el LSTM-AE aporta más valor al
pipeline completo de detección + explicación.

## 11. Evaluación en producción

`df_prod` contiene los últimos 6 días del dataset (12/06/2020 - 17/06/2020) que el modelo **nunca ha visto** durante entrenamiento ni validación. Esta evaluación confirma que las métricas de validación no están sobreajustadas al split de val.
Se evalúan los tres modelos (LSTM-AE, IF, LR) sobre `df_prod` con las mismas métricas para mantener la comparativa completa.

In [ ]:
# LSTM-AE sobre df_prod (usa estrategia ganadora e incluye MLP head si es necesario)
prod_scaled = scaler.transform(df_prod[features])

mask_outage_prod = ((df_prod["IRRADIATION"] > sun_umbral) & (df_prod["AC_POWER"] == 0)).values

X_prod, y_outage_prod = create_sequences(
    df_prod, prod_scaled, TIME_STEPS, anomaly_mask=mask_outage_prod
)
y_outage_prod = np.asarray(y_outage_prod, dtype=bool)

X_prod_pred = model.predict(X_prod, verbose=0)
abs_errors_prod = np.abs(X_prod_pred - X_prod)

forecast_err_prod = _compute_forecast_errors(model, encoder, mlp_head, X_prod, features)
prod_scores = _score_windows(abs_errors_prod, features, best_strategy, forecast_err_prod)

n_prod_windows = len(X_prod)
n_prod_outage = int(y_outage_prod.sum())
print(
    f"df_prod: {len(df_prod)} filas | {n_prod_windows} ventanas | {n_prod_outage} anomalías ({n_prod_outage / max(n_prod_windows, 1) * 100:.1f}%)"
)

roc_prod = roc_auc_score(y_outage_prod, prod_scores)
prec_prod, rec_prod, thr_prod = precision_recall_curve(y_outage_prod, prod_scores)
pr_prod = auc(rec_prod, prec_prod)

f1_prod = 2 * prec_prod[:-1] * rec_prod[:-1] / (prec_prod[:-1] + rec_prod[:-1] + 1e-12)
best_prod_idx = np.argmax(f1_prod)

# Evaluación con umbral fijado en validación (simula producción real)
_yp = prod_scores > optimal_threshold
_tp = int(np.sum(_yp & y_outage_prod))
_fp = int(np.sum(_yp & ~y_outage_prod))
_fn = int(np.sum(~_yp & y_outage_prod))
_tn = int(np.sum(~_yp & ~y_outage_prod))
_pr = _tp / (_tp + _fp) if (_tp + _fp) > 0 else 0.0
_rc = _tp / (_tp + _fn) if (_tp + _fn) > 0 else 0.0
_f1 = 2 * _pr * _rc / (_pr + _rc) if (_pr + _rc) > 0 else 0.0
metrics_prod_fixed = {
    "precision": _pr,
    "recall": _rc,
    "f1": _f1,
    "tp": _tp,
    "fp": _fp,
    "tn": _tn,
    "fn": _fn,
}
print(
    f"\n[LSTM-AE prod estrategia={best_strategy} umbral val={optimal_threshold:.4f}] TP={_tp} FP={_fp} TN={_tn} FN={_fn}"
)
print(f" - Precisión: {_pr:.3f} | Recall: {_rc:.3f} | F1: {_f1:.3f}")

print(f"\nLSTM-AE ({best_strategy}) sobre df_prod:")
print(f" - ROC-AUC: {roc_prod:.4f} | PR-AUC: {pr_prod:.4f}")
print(f" - F1-óptimo (recalculado): {f1_prod[best_prod_idx]:.4f}")
print(f" - F1 con umbral fijo de val ({optimal_threshold:.4f}): {metrics_prod_fixed['f1']:.4f}")

In [ ]:
# Isolation Forest sobre df_prod
prod_full_scaled = scaler.transform(df_prod[features])
if_scores_prod = -iso_forest.decision_function(prod_full_scaled)

# Etiquetas por fila
mask_outage_prod_rows = ((df_prod["IRRADIATION"] > sun_umbral) & (df_prod["AC_POWER"] == 0)).values

roc_if_prod = roc_auc_score(mask_outage_prod_rows, if_scores_prod)
prec_if_prod, rec_if_prod, _ = precision_recall_curve(mask_outage_prod_rows, if_scores_prod)
pr_if_prod = auc(rec_if_prod, prec_if_prod)

f1_if_prod = (
    2 * prec_if_prod[:-1] * rec_if_prod[:-1] / (prec_if_prod[:-1] + rec_if_prod[:-1] + 1e-12)
)
best_if_prod_idx = np.argmax(f1_if_prod)

print("\nIsolation Forest sobre df_prod:")
print(f" - ROC-AUC: {roc_if_prod:.4f} | PR-AUC: {pr_if_prod:.4f}")
print(f" - F1-óptimo: {f1_if_prod[best_if_prod_idx]:.4f}")

In [ ]:
# Regresión Lineal sobre df_prod
X_lr_prod = df_prod[lr_features].values
y_lr_prod = df_prod["AC_POWER"].values
lr_scores_prod = -(y_lr_prod - lr_model.predict(X_lr_prod))

roc_lr_prod = roc_auc_score(mask_outage_prod_rows, lr_scores_prod)
prec_lr_prod, rec_lr_prod, _ = precision_recall_curve(mask_outage_prod_rows, lr_scores_prod)
pr_lr_prod = auc(rec_lr_prod, prec_lr_prod)

f1_lr_prod = (
    2 * prec_lr_prod[:-1] * rec_lr_prod[:-1] / (prec_lr_prod[:-1] + rec_lr_prod[:-1] + 1e-12)
)
best_lr_prod_idx = np.argmax(f1_lr_prod)

print("\nRegresión Lineal sobre df_prod:")
print(f" - ROC-AUC: {roc_lr_prod:.4f}  |  PR-AUC: {pr_lr_prod:.4f}")
print(f" - F1-óptimo: {f1_lr_prod[best_lr_prod_idx]:.4f}")

### 11.1 Comparativa val vs prod

> **Nota:** Si obtenemos métricas de prod similares a val, no hay sobreajuste. Si prod cae significativamente, el umbral de val está demasiado afinado.

In [ ]:
print("=" * 90)
print("Comparativa validación vs producción (flag_outage)")
print("=" * 90)

header = f"\n{'Modelo':<25} {'Split':<7} {'ROC-AUC':>8} {'PR-AUC':>8} {'F1':>8}"
print(header)
print("-" * 90)
print(
    f"{'Regresión Lineal':<25} {'val':<7} {roc_lr:>8.4f} {pr_lr:>8.4f} {f1_lr[best_lr_idx]:>8.4f}"
)
print(
    f"{'':<25} {'prod':<7} {roc_lr_prod:>8.4f} {pr_lr_prod:>8.4f} {f1_lr_prod[best_lr_prod_idx]:>8.4f}"
)
print(
    f"{'Isolation Forest':<25} {'val':<7} {roc_if:>8.4f} {pr_if:>8.4f} {f1_if[best_if_idx]:>8.4f}"
)
print(
    f"{'':<25} {'prod':<7} {roc_if_prod:>8.4f} {pr_if_prod:>8.4f} {f1_if_prod[best_if_prod_idx]:>8.4f}"
)
print(
    f"{'LSTM-AE (D_p95_power)':<25} {'val':<7} {roc_auc_val:>8.4f} {pr_auc_val:>8.4f} {metrics_optimal['f1']:>8.4f}"
)
print(f"{'':<25} {'prod':<7} {roc_prod:>8.4f} {pr_prod:>8.4f} {f1_prod[best_prod_idx]:>8.4f}")
print("-" * 90)

In [ ]:
# Curvas PR: val vs prod para LSTM-AE
fig, axes = plt.subplots(1, 2, figsize=(18, 7))

# Panel izquierdo
# LSTM-AE val vs prod
axes[0].plot(
    rec_curve,
    prec_curve,
    color="steelblue",
    linewidth=2,
    label=f"Val (PR-AUC={pr_auc_val:.4f})",
)
axes[0].plot(
    rec_prod,
    prec_prod,
    color="coral",
    linewidth=2,
    linestyle="--",
    label=f"Prod (PR-AUC={pr_prod:.4f})",
)
axes[0].set_xlabel("Recall", fontsize=12)
axes[0].set_ylabel("Precisión", fontsize=12)
axes[0].set_title("LSTM-AE (D_p95_power): Val vs Prod", fontsize=13)
axes[0].legend(fontsize=11)
axes[0].grid(alpha=0.3)

# Panel derecho
# Los 3 modelos sobre prod
axes[1].plot(
    rec_prod,
    prec_prod,
    color="steelblue",
    linewidth=2,
    label=f"LSTM-AE (PR-AUC={pr_prod:.4f})",
)
axes[1].plot(
    rec_if_prod,
    prec_if_prod,
    color="darkorange",
    linewidth=2,
    linestyle="--",
    label=f"Isolation Forest (PR-AUC={pr_if_prod:.4f})",
)
axes[1].plot(
    rec_lr_prod,
    prec_lr_prod,
    color="forestgreen",
    linewidth=2,
    linestyle=":",
    label=f"Regresión Lineal (PR-AUC={pr_lr_prod:.4f})",
)
axes[1].set_xlabel("Recall", fontsize=12)
axes[1].set_ylabel("Precisión", fontsize=12)
axes[1].set_title("Comparativa sobre df_prod (flag_outage)", fontsize=13)
axes[1].legend(fontsize=11)
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

## 12. Relevancia de modelado *per-plant*

Vale la pena evaluar el entrenamiento de 2 modelos especializados (uno por planta) en lugar de un modelo combinado. Las diferencias térmicas y de escala DC entre las plantas son factores críticos que podrían validar esta estrategia.

**Procedimiento:** 3 semillas × {combinado, per-plant × 2 plantas} + evaluación cruzada (P1-->P2, P2-->P1) + métrica agregada routed. Decisión por ganancia de PR-AUC (umbral 5%).

### 12.1 Preparación de splits por planta

In [ ]:
# Prepara splits temporales y máscaras de limpieza
# Si plant_id=None procesa el df completo (modo combinado)
def prepare_plant_data(df_source, plant_id=None, train_frac=0.60, val_frac=0.20):
    df = df_source.copy()
    # Corrección de escala DC Planta 1 (df_full conserva la escala original)
    df.loc[df["PLANT"] == 1, "DC_POWER"] = df.loc[df["PLANT"] == 1, "DC_POWER"] / 10

    if plant_id is not None:
        df = df[df["PLANT"] == plant_id].copy()

    unique_dates_local = df["DATE_TIME"].dt.date.sort_values().unique()
    n = len(unique_dates_local)
    idx_t = int(n * train_frac)
    idx_v = int(n * (train_frac + val_frac))
    train_end = unique_dates_local[idx_t]
    val_end = unique_dates_local[idx_v]

    df_tr = df[df["DATE_TIME"].dt.date <= train_end].copy()
    df_va = df[(df["DATE_TIME"].dt.date > train_end) & (df["DATE_TIME"].dt.date <= val_end)].copy()
    df_pr = df[df["DATE_TIME"].dt.date > val_end].copy()

    df_tr_h = df_tr[make_healthy_mask(df_tr)].copy()
    df_va_h = df_va[make_healthy_mask(df_va)].copy()

    return {
        "train_healthy": df_tr_h,
        "val_healthy": df_va_h,
        "val_full": df_va,
        "prod": df_pr,
        "split_dates": {"train_end": str(train_end), "val_end": str(val_end)},
        "plant_id": plant_id,
    }

### 12.2 Baseline combinado (3 seeds)

In [ ]:
SEEDS = [42, 123, 2026]
results = {"combined": {}, "per_plant": {1: {}, 2: {}}, "cross": {}, "routed": {}}

for s in SEEDS:
    print(f"\n{'=' * 60}\n[COMBINED] seed={s}\n{'=' * 60}")
    splits = prepare_plant_data(df_full, plant_id=None)
    mdl, enc, head, scl, _ = train_model(
        splits["train_healthy"],
        splits["val_healthy"],
        features,
        TIME_STEPS,
        seed=s,
        verbose=1,
    )
    thr, _ = compute_threshold(
        mdl,
        scl,
        splits["val_full"],
        features,
        TIME_STEPS,
        scoring=best_strategy,
        encoder_fit=enc,
        head_fit=head,
    )
    metrics = evaluate(
        mdl,
        scl,
        thr,
        splits["val_full"],
        features,
        TIME_STEPS,
        scoring=best_strategy,
        encoder_fit=enc,
        head_fit=head,
    )
    results["combined"][s] = {
        "metrics": {k: v for k, v in metrics.items() if k not in ("scores", "y_outage")},
        "model": mdl,
        "encoder": enc,
        "mlp_head": head,
        "scaler": scl,
        "threshold": thr,
        "splits": splits,
    }
    print(
        f"[COMBINED s={s}] ROC-AUC={metrics['roc_auc']:.4f} PR-AUC={metrics['pr_auc']:.4f} F1={metrics['f1']:.4f}"
    )


# Agregación media±std
def _agg(run_dict, metric):
    vals = [r["metrics"][metric] for r in run_dict.values()]
    return float(np.mean(vals)), float(np.std(vals))


for m in ["roc_auc", "pr_auc", "f1"]:
    mu, sd = _agg(results["combined"], m)
    print(f"[COMBINED agg] {m}: {mu:.4f} ± {sd:.4f}")

### 12.3 *Per-plant* (3 seeds × 2 plantas)

In [ ]:
for pid in [1, 2]:
    for s in SEEDS:
        print(f"\n{'=' * 60}\n[PLANT={pid}] seed={s}\n{'=' * 60}")
        splits_p = prepare_plant_data(df_full, plant_id=pid)
        mdl_p, enc_p, head_p, scl_p, _ = train_model(
            splits_p["train_healthy"],
            splits_p["val_healthy"],
            features,
            TIME_STEPS,
            seed=s,
            verbose=1,
        )
        thr_p, _ = compute_threshold(
            mdl_p,
            scl_p,
            splits_p["val_full"],
            features,
            TIME_STEPS,
            scoring=best_strategy,
            encoder_fit=enc_p,
            head_fit=head_p,
        )
        metrics_p = evaluate(
            mdl_p,
            scl_p,
            thr_p,
            splits_p["val_full"],
            features,
            TIME_STEPS,
            scoring=best_strategy,
            encoder_fit=enc_p,
            head_fit=head_p,
        )
        results["per_plant"][pid][s] = {
            "metrics": {k: v for k, v in metrics_p.items() if k not in ("scores", "y_outage")},
            "model": mdl_p,
            "encoder": enc_p,
            "mlp_head": head_p,
            "scaler": scl_p,
            "threshold": thr_p,
            "splits": splits_p,
        }
        print(
            f"[PLANT={pid} s={s}] ROC-AUC={metrics_p['roc_auc']:.4f} PR-AUC={metrics_p['pr_auc']:.4f} F1={metrics_p['f1']:.4f}"
        )

for pid in [1, 2]:
    for m in ["roc_auc", "pr_auc", "f1"]:
        mu, sd = _agg(results["per_plant"][pid], m)
        print(f"[PLANT={pid} agg] {m}: {mu:.4f} ± {sd:.4f}")

### 12.4 Evaluación cruzada

In [ ]:
# Modelo Planta 1 sobre datos Planta 2 y viceversa. Usa seed mediano por PR-AUC
def _median_seed(run_dict, metric="pr_auc"):
    vals = [(s, r["metrics"][metric]) for s, r in run_dict.items()]
    vals.sort(key=lambda x: x[1])
    return vals[len(vals) // 2][0]


s1 = _median_seed(results["per_plant"][1])
s2 = _median_seed(results["per_plant"][2])
print(f"Seeds mediano: P1={s1}, P2={s2}")

# P1 sobre datos P2
m_p1 = results["per_plant"][1][s1]
splits_p2 = results["per_plant"][2][s2]["splits"]
metrics_p1_on_p2 = evaluate(
    m_p1["model"],
    m_p1["scaler"],
    m_p1["threshold"],
    splits_p2["val_full"],
    features,
    TIME_STEPS,
    scoring=best_strategy,
    encoder_fit=m_p1["encoder"],
    head_fit=m_p1["mlp_head"],
)
results["cross"]["p1_on_p2"] = {
    k: v for k, v in metrics_p1_on_p2.items() if k not in ("scores", "y_outage")
}
print(
    f"[CROSS P1-->P2] ROC-AUC={metrics_p1_on_p2['roc_auc']:.4f} PR-AUC={metrics_p1_on_p2['pr_auc']:.4f}"
)

# P2 sobre datos P1
m_p2 = results["per_plant"][2][s2]
splits_p1 = results["per_plant"][1][s1]["splits"]
metrics_p2_on_p1 = evaluate(
    m_p2["model"],
    m_p2["scaler"],
    m_p2["threshold"],
    splits_p1["val_full"],
    features,
    TIME_STEPS,
    scoring=best_strategy,
    encoder_fit=m_p2["encoder"],
    head_fit=m_p2["mlp_head"],
)
results["cross"]["p2_on_p1"] = {
    k: v for k, v in metrics_p2_on_p1.items() if k not in ("scores", "y_outage")
}
print(
    f"[CROSS P2-->P1] ROC-AUC={metrics_p2_on_p1['roc_auc']:.4f} PR-AUC={metrics_p2_on_p1['pr_auc']:.4f}"
)

### Per-plant routed (métrica agregada)

In [ ]:
# Cada ventana se enruta al modelo de su planta. Métricas agregadas comparables 1 a 1 contra combinado.
routed_scores, routed_y = [], []
for pid in [1, 2]:
    sseed = s1 if pid == 1 else s2
    entry = results["per_plant"][pid][sseed]
    ev = evaluate(
        entry["model"],
        entry["scaler"],
        entry["threshold"],
        entry["splits"]["val_full"],
        features,
        TIME_STEPS,
        scoring=best_strategy,
        encoder_fit=entry["encoder"],
        head_fit=entry["mlp_head"],
    )
    routed_scores.append(ev["scores"])
    routed_y.append(ev["y_outage"])

all_scores = np.concatenate(routed_scores)
all_y = np.concatenate(routed_y)

# Media simple de umbrales per-plant
thr_routed = float(
    np.mean(
        [
            results["per_plant"][1][s1]["threshold"],
            results["per_plant"][2][s2]["threshold"],
        ]
    )
)

y_pred = all_scores > thr_routed
tp = int(np.sum(y_pred & all_y))
fp = int(np.sum(y_pred & ~all_y))
fn = int(np.sum(~y_pred & all_y))
tn = int(np.sum(~y_pred & ~all_y))
prec_r = tp / (tp + fp) if (tp + fp) > 0 else 0.0
rec_r = tp / (tp + fn) if (tp + fn) > 0 else 0.0
f1_r = 2 * prec_r * rec_r / (prec_r + rec_r) if (prec_r + rec_r) > 0 else 0.0
roc_r = float(roc_auc_score(all_y, all_scores))
prec_c, rec_c, _ = precision_recall_curve(all_y, all_scores)
pr_r = float(auc(rec_c, prec_c))

results["routed"] = {
    "roc_auc": roc_r,
    "pr_auc": pr_r,
    "f1": f1_r,
    "precision": prec_r,
    "recall": rec_r,
    "tp": tp,
    "fp": fp,
    "tn": tn,
    "fn": fn,
    "threshold": thr_routed,
    "note": "umbral = media simple de umbrales per-plant",
}
print(f"[ROUTED] ROC-AUC={roc_r:.4f} PR-AUC={pr_r:.4f} F1={f1_r:.4f}")

### 12.6 Tabla comparativa y decisión

In [ ]:
rows = []
for label, agg_dict in [
    ("Combinado", results["combined"]),
    ("Per-plant P1 (aislado)", results["per_plant"][1]),
    ("Per-plant P2 (aislado)", results["per_plant"][2]),
]:
    roc_m, roc_s = _agg(agg_dict, "roc_auc")
    pr_m, pr_s = _agg(agg_dict, "pr_auc")
    f1_m, f1_s = _agg(agg_dict, "f1")
    rows.append(
        {
            "Estrategia": label,
            "ROC-AUC": f"{roc_m:.4f} ± {roc_s:.4f}",
            "PR-AUC": f"{pr_m:.4f} ± {pr_s:.4f}",
            "F1": f"{f1_m:.4f} ± {f1_s:.4f}",
        }
    )

rows.append(
    {
        "Estrategia": "Per-plant (routed)",
        "ROC-AUC": f"{results['routed']['roc_auc']:.4f}",
        "PR-AUC": f"{results['routed']['pr_auc']:.4f}",
        "F1": f"{results['routed']['f1']:.4f}",
    }
)
rows.append(
    {
        "Estrategia": "Cross P1-->P2",
        "ROC-AUC": f"{results['cross']['p1_on_p2']['roc_auc']:.4f}",
        "PR-AUC": f"{results['cross']['p1_on_p2']['pr_auc']:.4f}",
        "F1": f"{results['cross']['p1_on_p2']['f1']:.4f}",
    }
)
rows.append(
    {
        "Estrategia": "Cross P2-->P1",
        "ROC-AUC": f"{results['cross']['p2_on_p1']['roc_auc']:.4f}",
        "PR-AUC": f"{results['cross']['p2_on_p1']['pr_auc']:.4f}",
        "F1": f"{results['cross']['p2_on_p1']['f1']:.4f}",
    }
)

df_comparison = pd.DataFrame(rows)
print(df_comparison.to_string(index=False))

combined_pr = _agg(results["combined"], "pr_auc")[0]
routed_pr = results["routed"]["pr_auc"]
rel_gain = (routed_pr - combined_pr) / max(combined_pr, 1e-9)
print(f"\nGanancia relativa per-plant routed vs combinado (PR-AUC): {rel_gain * 100:+.2f}%")

## 13. Exportación final

Contrato de salida **por planta** (cuatro archivos en `models/keras/plant_{1|2}/*`):

- `model_lstm_solar.keras` → pesos + arquitectura.
- `model_mlp_head.keras` → cabeza MLP para scoring híbrido.
- `scaler_solar.pkl` → MinMaxScaler ajustado.
- `config_solar.json` → umbral, features, ventana, métricas, trazabilidad.


In [ ]:
# Importante: suponemos que se ha ejecutado en Google Colab

OUTPUT_MODELS = os.path.join("..", "models", "keras")
OUTPUT_DATA = os.path.join("..", "data")
DRIVE_MODELS = "/content/drive/MyDrive/msc-muia-2026/models/keras"
DRIVE_DATA = "/content/drive/MyDrive/msc-muia-2026/data"

for pid in [1, 2]:
    os.makedirs(os.path.join(OUTPUT_MODELS, f"plant_{pid}"), exist_ok=True)
os.makedirs(OUTPUT_DATA, exist_ok=True)


def _sha256_file(file_path):
    h = hashlib.sha256()
    with open(file_path, "rb") as f:
        for chunk in iter(lambda: f.read(8192), b""):
            h.update(chunk)
    return h.hexdigest()


dataset_hashes = {f: _sha256_file(os.path.join(path, f)) for f in available_datasets}
print("Dataset SHA256 (trazabilidad):")
for k, v in dataset_hashes.items():
    print(f" - {k}: {v[:16]}...")

### 13.1 Selección del ganador por planta

Para cada planta se evalúan todos los runs entrenados sobre sus datos (seeds 42, 123, 2026)
y se elige el de mayor PR-AUC sobre la validación de esa planta.
El modelo combinado queda en `results` como referencia comparativa pero no se exporta.


In [ ]:
winners = {}
for pid in [1, 2]:
    plant_candidates = []
    for seed_, run in results["per_plant"][pid].items():
        plant_candidates.append(
            {
                "label": f"plant{pid}_s{seed_}",
                "scope": f"plant_{pid}",
                "seed": seed_,
                "model": run["model"],
                "encoder": run["encoder"],
                "mlp_head": run["mlp_head"],
                "scaler": run["scaler"],
                "threshold": run["threshold"],
                "metrics": run["metrics"],
                "splits": run["splits"],
            }
        )
    best = max(plant_candidates, key=lambda c: c["metrics"]["pr_auc"])
    winners[pid] = best
    print(f"[PLANT {pid}] Ranking por PR-AUC:")
    for c in sorted(plant_candidates, key=lambda x: x["metrics"]["pr_auc"], reverse=True):
        mark = "*" if c["label"] == best["label"] else " "
        print(
            f"  {mark} {c['label']:<16} PR-AUC={c['metrics']['pr_auc']:.4f}"
            f"  ROC-AUC={c['metrics']['roc_auc']:.4f}  F1={c['metrics']['f1']:.4f}"
        )
    print(f"  => WINNER P{pid}: {best['label']} (seed={best['seed']})")

### 13.2 Escritura del contrato

In [ ]:
def _to_json(obj):
    if isinstance(obj, np.ndarray):
        return obj.tolist()
    if hasattr(obj, "item"):
        return obj.item()
    return float(obj)


def export_winner(plant_id, w):
    out_dir = os.path.join(OUTPUT_MODELS, f"plant_{plant_id}")
    os.makedirs(out_dir, exist_ok=True)

    w["model"].save(os.path.join(out_dir, "model_lstm_solar.keras"))
    w["mlp_head"].save(os.path.join(out_dir, "model_mlp_head.keras"))
    joblib.dump(w["scaler"], os.path.join(out_dir, "scaler_solar.pkl"))

    config_payload = {
        "threshold": float(w["threshold"]),
        "sun_threshold": float(sun_umbral),
        "time_steps": int(TIME_STEPS),
        "features": list(features),
        "scoring_strategy": str(best_strategy),
        "scoring_power_idx": [features.index("DC_POWER"), features.index("AC_POWER")],
        "mlp_head_target_idx": features.index("AC_POWER"),
        "bottleneck_layer_idx": 3,
        "training_scope": str(w["scope"]),
        "training_seed": int(w["seed"]),
        "dataset_sha256": dataset_hashes,
        "metrics": {
            k: v.tolist()
            if isinstance(v, np.ndarray)
            else v
            if isinstance(v, (str, bool))
            else float(v)
            for k, v in w["metrics"].items()
        },
    }
    with open(os.path.join(out_dir, "config_solar.json"), "w") as f:
        json.dump(config_payload, f, indent=4, default=_to_json)

    print(f"[EXPORT P{plant_id}] artefactos guardados en {out_dir}")
    for fname in [
        "model_lstm_solar.keras",
        "model_mlp_head.keras",
        "scaler_solar.pkl",
        "config_solar.json",
    ]:
        assert os.path.exists(os.path.join(out_dir, fname)), f"FALTA: {fname}"
    print(f"[EXPORT P{plant_id}] verificación OK (4 archivos presentes)")


for pid in [1, 2]:
    export_winner(pid, winners[pid])

df_prod.to_csv(os.path.join(OUTPUT_DATA, "dataset_prod.csv"), index=False)

In [ ]:
# Evaluación final sobre producción
print("=" * 70)
print("Evaluación en producción — ganadores per-plant (routed)")
print("=" * 70)

prod_routed_scores, prod_routed_y = [], []
for pid in [1, 2]:
    w = winners[pid]
    ev = evaluate(
        w["model"],
        w["scaler"],
        w["threshold"],
        w["splits"]["prod"],
        features,
        TIME_STEPS,
        scoring=best_strategy,
        encoder_fit=w["encoder"],
        head_fit=w["mlp_head"],
    )
    prod_routed_scores.append(ev["scores"])
    prod_routed_y.append(ev["y_outage"])
    print(
        f"[{w['label']}] prod \u2192 ROC-AUC={ev['roc_auc']:.4f} | PR-AUC={ev['pr_auc']:.4f} | F1={ev['f1']:.4f}"
    )

# Métricas routed agregadas sobre producción
all_prod_scores = np.concatenate(prod_routed_scores)
all_prod_y = np.concatenate(prod_routed_y)
thr_prod_routed = float(np.mean([winners[1]["threshold"], winners[2]["threshold"]]))
roc_prod_r = float(roc_auc_score(all_prod_y, all_prod_scores))
prec_c, rec_c, _ = precision_recall_curve(all_prod_y, all_prod_scores)
pr_prod_r = float(auc(rec_c, prec_c))
y_pred_prod = all_prod_scores > thr_prod_routed
tp = int(np.sum(y_pred_prod & all_prod_y))
fp = int(np.sum(y_pred_prod & ~all_prod_y))
fn = int(np.sum(~y_pred_prod & all_prod_y))
tn = int(np.sum(~y_pred_prod & ~all_prod_y))
prec_r = tp / (tp + fp) if (tp + fp) > 0 else 0.0
rec_r = tp / (tp + fn) if (tp + fn) > 0 else 0.0
f1_prod_r = 2 * prec_r * rec_r / (prec_r + rec_r) if (prec_r + rec_r) > 0 else 0.0

print(f"\n[ROUTED PROD] ROC-AUC={roc_prod_r:.4f} | PR-AUC={pr_prod_r:.4f} | F1={f1_prod_r:.4f}")

# Comparativa val vs prod
print("\nComparativa val vs prod (ganadores enrutados):")
print(
    f"  Val  \u2192 ROC-AUC={results['routed']['roc_auc']:.4f} | PR-AUC={results['routed']['pr_auc']:.4f} | F1={results['routed']['f1']:.4f}"
)
print(f"  Prod \u2192 ROC-AUC={roc_prod_r:.4f} | PR-AUC={pr_prod_r:.4f} | F1={f1_prod_r:.4f}")
delta_pr = pr_prod_r - results["routed"]["pr_auc"]
print(f"  \u0394 PR-AUC: {delta_pr:+.4f}")

### 13.3 Sincronización con Drive y verificación de carga

In [ ]:
if IN_COLAB:
    drive.mount("/content/drive")
    os.makedirs(DRIVE_DATA, exist_ok=True)

    for pid in [1, 2]:
        src_dir = os.path.join(OUTPUT_MODELS, f"plant_{pid}")
        dst_dir = os.path.join(DRIVE_MODELS, f"plant_{pid}")
        os.makedirs(dst_dir, exist_ok=True)
        for fname in [
            "model_lstm_solar.keras",
            "model_mlp_head.keras",
            "scaler_solar.pkl",
            "config_solar.json",
        ]:
            shutil.copy(os.path.join(src_dir, fname), os.path.join(dst_dir, fname))
        print(f"[DRIVE P{pid}] copia OK en {dst_dir}")

    shutil.copy(
        os.path.join(OUTPUT_DATA, "dataset_prod.csv"),
        os.path.join(DRIVE_DATA, "dataset_prod.csv"),
    )
    print("[DRIVE] dataset_prod.csv OK")